# Joint Distributions

ProbPipe provides several joint distribution types for modeling multiple related quantities:

| Type | Use Case | Key Feature |
|------|----------|-------------|
| `ProductDistribution` | Independent named components | `log_prob` = sum of marginals |
| `SequentialJointDistribution` | Autoregressive dependence | Callables receive earlier samples |
| `JointEmpirical` | MCMC / importance samples | Joint row resampling preserves correlation |
| `JointGaussian` | Analytical Gaussian models | Exact conditioning via `condition_on()` |

All joint distributions inherit from `PyTreeArrayDistribution` and return **pytrees** (dicts) from `sample()`.
Components can be organized as **flat dicts** or **nested dicts** — the API works identically either way.

This tutorial walks through the flat-dict API first, then shows how nested dicts generalize it.

---
## Class Hierarchy

```
Distribution
└── JointDistribution          ← generic base
    ├── ProductDistribution    ← generic independent-marginals marker
    └── JointArrayDistribution ← JAX-backed; adds pytree shape semantics
        ├── ProductArrayDistribution     ← JAX independent joint (was ProductDistribution)
        ├── SequentialJointDistribution  ← autoregressive
        ├── JointGaussian                ← exact Gaussian conditioning
        └── JointEmpirical               ← weighted particle approximation
```

**Migration note:** `ProductDistribution` is now the *generic* (non-JAX) base class.
Code that previously instantiated `ProductDistribution(x=..., y=...)` should now
use `ProductArrayDistribution(x=..., y=...)` for JAX-backed distributions.
`isinstance(x, ProductDistribution)` still returns `True` for `ProductArrayDistribution`
instances, so upstream `isinstance` checks remain valid.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from probpipe import (
    Normal,
    Gamma,
    MultivariateNormal,
    JointDistribution,
    ProductDistribution,
    JointArrayDistribution,
    ProductArrayDistribution,
    SequentialJointDistribution,
    JointEmpirical,
    JointGaussian,
    DistributionView,
    EmpiricalDistribution,
    FlattenedView,
)
from probpipe.core.node import workflow_function
from probpipe.core.provenance import provenance_ancestors
from probpipe import condition_on, log_prob, mean, sample, variance

---
## Part 1: Flat Dict — ProductArrayDistribution

The simplest joint distribution is a `ProductArrayDistribution` with flat keyword arguments.
Each keyword becomes a component name, and the corresponding value is an `ArrayDistribution`.

> **Hierarchy note (issue #92):** `ProductArrayDistribution` is the JAX-backed implementation
> of the generic `ProductDistribution` base. Use `ProductArrayDistribution` when constructing
> concrete joint distributions. `isinstance(x, ProductDistribution)` still returns `True` for
> both the generic base and the JAX implementation.

### Construction and Introspection

In [ ]:
joint = ProductArrayDistribution(
    theta=Normal(loc=0.0, scale=1.0),
    sigma=Normal(loc=1.0, scale=0.5),
)
print(f"Joint: {joint}")
print(f"Components: {joint.component_names}")
print(f"Event shapes: {joint.event_shapes}")
print(f"Event size: {joint.event_size}")
print(f"Mean: {mean(joint)}")
print(f"Variance: {variance(joint)}")

### Sampling

`sample()` returns a dict of per-component arrays. Each value has shape
`(*sample_shape, *batch_shape, *component_event_shape)`.

In [ ]:
samples = sample(joint, sample_shape=(5,))
print(f"Sample keys: {list(samples.keys())}")
print(f"theta samples: {samples['theta']}")
print(f"sigma samples: {samples['sigma']}")

### Accessing Components

`joint['name']` returns a `DistributionView` — a lightweight `ArrayDistribution`
that remembers its parent joint. When sampled standalone, it draws a full joint
sample and extracts the requested component.

In [ ]:
view_theta = joint['theta']
print(f"View: {view_theta}")
print(f"Event shape: {view_theta.event_shape}")
print(f"Mean: {float(mean(view_theta)):.3f}")
s = sample(view_theta, sample_shape=(5,))
print(f"Samples: {s}")

### Correlated Broadcasting

When two `DistributionView` instances from the **same parent** are passed to a
`workflow_function`, the parent is sampled **once** and each view extracts its
component. This preserves correlation. Compare the two cases below: jointly
sampled views yield `a - b ≈ 0` (since both come from the same sample), while
independent distributions do not.

In [ ]:
@workflow_function(n_broadcast_samples=200, vectorize='loop', seed=0)
def difference(a, b):
    return a - b

# Jointly sampled: both come from the same parent → a - b ≈ 0
result_joint = difference(**joint.bind(a='theta', b='theta'))
print(f"Joint std:  {float(jnp.std(result_joint.samples)):.4f}  (≈ 0)")

# Independent: different distributions → a - b has nonzero variance
result_indep = difference(a=Normal(0, 1), b=Normal(0, 1))
print(f"Indep std:  {float(jnp.std(result_indep.samples)):.4f}  (≈ √2)")

### Log-Probability

`log_prob()` accepts a dict with the same structure as the components.
For a `ProductDistribution`, it equals the sum of per-component log-probs.

In [ ]:
x = {"theta": jnp.array(0.5), "sigma": jnp.array(1.5)}
lp_joint = log_prob(joint, x)
lp_sum = log_prob(joint.components['theta'], 0.5) + log_prob(joint.components['sigma'], 1.5)
print(f"Joint log_prob: {float(lp_joint):.4f}")
print(f"Sum of marginals: {float(lp_sum):.4f}")
print(f"Equal: {bool(jnp.isclose(lp_joint, lp_sum))}")

`component_log_prob()` returns the per-component contributions as a dict:

In [ ]:
clp = joint.component_log_prob(x)
print(f"Component log-probs: {clp}")
print(f"Sum: {float(clp['theta'] + clp['sigma']):.4f}")

### Conditioning

`condition_on()` returns a new joint distribution over the remaining
(unconditioned) components.

In [ ]:
conditioned = condition_on(joint, theta=jnp.array(2.0))
print(f"Original components: {joint.component_names}")
print(f"After conditioning on theta=2.0: {conditioned.component_names}")
print(f"Event shapes: {conditioned.event_shapes}")

s_cond = sample(conditioned, sample_shape=(5,))
print(f"\nKeys: {list(s_cond.keys())}")
print(f"Sigma samples: {s_cond['sigma']}")

### Flat-Vector Interop

`flatten_value()` concatenates all component arrays into a single flat vector.
`unflatten_value()` reverses this. `as_flat_distribution()` returns a
`FlattenedView` wrapping the joint as a flat `ArrayDistribution`.

In [ ]:
s = sample(joint, sample_shape=(3,))
flat = joint.flatten_value(s)
print(f"Flat shape: {flat.shape}")
print(f"Flat:\n{flat}")

recovered = joint.unflatten_value(flat)
print(f"\nRecovered theta: {recovered['theta']}")
print(f"Recovered sigma: {recovered['sigma']}")

# FlattenedView for algorithms expecting flat vectors
flat_dist = joint.as_flat_distribution()
print(f"\nFlattenedView event_shape: {flat_dist.event_shape}")
print(f"FlattenedView sample: {sample(flat_dist, sample_shape=(2,))}")

### Multivariate Components

`ProductDistribution` works with multivariate components. The total
`event_size` is the sum of all component event sizes.

In [ ]:
joint_mv = ProductArrayDistribution(
    pos=MultivariateNormal(loc=jnp.zeros(2), cov=jnp.eye(2)),
    vel=MultivariateNormal(loc=jnp.array([1.0, 0.0]), cov=0.1 * jnp.eye(2)),
)
print(f"Event shapes: {joint_mv.event_shapes}")
print(f"Event size: {joint_mv.event_size}")

ss = sample(joint_mv, sample_shape=(500,))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(ss['pos'][:, 0], ss['pos'][:, 1], alpha=0.3, s=5)
ax1.set_title("Position"); ax1.axis("equal")
ax2.scatter(ss['vel'][:, 0], ss['vel'][:, 1], alpha=0.3, s=5, color="orange")
ax2.set_title("Velocity"); ax2.axis("equal")
plt.tight_layout(); plt.show()

---
## Part 2: Nested Dict — Organizing Components

Component distributions can be organized into **nested dicts** for
structure.  The nesting is purely organizational — all component
distributions remain statistically independent.  The entire API
generalizes seamlessly: samples, `event_shapes`, `log_prob`, etc.
all mirror the nested structure.

In [ ]:
nested = ProductArrayDistribution(
    physics={"force": Normal(loc=0.0, scale=1.0),
             "mass": Gamma(concentration=2.0, rate=1.0)},
    observation=Normal(loc=0.0, scale=0.1),
)
print(f"Joint: {nested}")
print(f"Event shapes: {nested.event_shapes}")
print(f"Event size: {nested.event_size}")
print(f"Component names (key paths): {nested.component_names}")

### Nested Sampling

Samples mirror the nested dict structure:

In [ ]:
s = sample(nested, sample_shape=(3,))
print(f"Top-level keys: {list(s.keys())}")
print(f"physics sub-keys: {list(s['physics'].keys())}")
print(f"\nforce samples:       {s['physics']['force']}")
print(f"mass samples:        {s['physics']['mass']}")
print(f"observation samples: {s['observation']}")

### Nested Log-Prob and Moments

`log_prob()` accepts the same nested structure. `mean()` and `variance()`
return nested dicts. `component_log_prob()` returns per-leaf contributions.

In [ ]:
s = sample(nested)
print(f"log_prob: {float(log_prob(nested, s)):.4f}")
print(f"\nComponent log-probs: {nested.component_log_prob(s)}")
print(f"\nMean: {mean(nested)}")
print(f"Variance: {variance(nested)}")

### Accessing Nested Leaves

Use a **key-path tuple** to reach a nested leaf. For top-level leaves,
a plain string still works.

In [ ]:
# Nested leaf: use a tuple of strings
view_force = nested["physics", "force"]
print(f"Force view: {view_force}")
print(f"Force event_shape: {view_force.event_shape}")
print(f"Force samples: {sample(view_force, sample_shape=(5,))}")

# Top-level leaf: plain string
view_obs = nested["observation"]
print(f"\nObservation view: {view_obs}")
print(f"Observation samples: {sample(view_obs, sample_shape=(5,))}")

### Accessing Internal Nodes (Sub-Joints)

When a key path resolves to an **intermediate dict node** (not a
component distribution), `__getitem__` returns a new
`ProductDistribution` wrapping that sub-tree.  This is the
**marginal** distribution over the component distributions in
that sub-tree.

In [ ]:
# Indexing an internal node returns a sub-joint
sub_physics = nested["physics"]
print(f"Type: {type(sub_physics).__name__}")
print(f"Components: {sub_physics.component_names}")
print(f"Event shapes: {sub_physics.event_shapes}")
print(f"Event size: {sub_physics.event_size}")

# The sub-joint samples and computes log-prob like any ProductDistribution
s = sample(sub_physics, sample_shape=(3,))
print(f"\nSample keys: {list(s.keys())}")
print(f"Force samples: {s['force']}")
print(f"Mass samples:  {s['mass']}")
print(f"Log-prob: {log_prob(sub_physics, sample(sub_physics))}")
print(f"Mean: {mean(sub_physics)}")

### Nested Broadcasting

Views from nested joints work in `workflow_function` broadcasting just like
flat views — correlated components from the same parent are sampled jointly.

In [ ]:
@workflow_function(n_broadcast_samples=200, vectorize='loop', seed=1)
def scale_force(force, mass):
    """Newton's F = m*a  →  a = force / mass"""
    return force / mass

result = scale_force(
    force=nested["physics", "force"],
    mass=nested["physics", "mass"],
)
output = result
print(f"Result type: {type(result).__name__}")
print(f"Acceleration mean: {float(jnp.mean(output.samples)):.3f}")
print(f"Acceleration std:  {float(jnp.std(output.samples)):.3f}")

### Nested Flatten / Unflatten

`flatten_value()` and `unflatten_value()` work on nested structures,
flattening all leaves into a single vector and reconstructing the
nested dict on the way back.

In [ ]:
s = sample(nested, sample_shape=(4,))
flat = nested.flatten_value(s)
print(f"Flat shape: {flat.shape}  (3 scalar leaves → event_size=3)")

recovered = nested.unflatten_value(flat)
print(f"Recovered physics.force: {recovered['physics']['force']}")
print(f"Recovered observation:   {recovered['observation']}")

# Verify roundtrip
for orig, rec in zip(jax.tree.leaves(s), jax.tree.leaves(recovered)):
    assert jnp.allclose(orig, rec)
print("\n✓ Roundtrip verified")

### Conditioning on Nested Components

`condition_on()` works at the **component distribution** level — you
specify observed values for individual component distributions.
You can condition on a single component, or on multiple components
at once.  If all component distributions under an intermediate dict
are conditioned, that dict is automatically pruned from the result.

Three equivalent calling conventions are supported:

1. **Kwargs with nested dicts** (most common): `condition_on(physics={"force": val})`
2. **Positional dict**: `condition_on({"physics": {"force": val}})`
3. **Flat kwargs** for top-level components: `condition_on(observation=val)`

In [ ]:
# Condition on a single nested component
cond1 = condition_on(nested, physics={"force": jnp.array(1.0)})
print("=== Condition on physics > force ===")
print(f"Remaining components: {cond1.component_names}")
print(f"Event size: {cond1.event_size}")
s1 = sample(cond1, sample_shape=(3,))
print(f"Sample keys: {list(s1.keys())}")
print(f"Physics sub-keys: {list(s1['physics'].keys())}")
print(f"Mass samples: {s1['physics']['mass']}")
print(f"Observation samples: {s1['observation']}")

In [ ]:
# Condition on ALL components under physics — the dict is pruned entirely
cond2 = condition_on(nested,
    physics={"force": jnp.array(1.0), "mass": jnp.array(2.0)}
)
print("=== Condition on both physics components ===")
print(f"Remaining components: {cond2.component_names}")
print(f"Event size: {cond2.event_size}")
print(f"Sample: {sample(cond2)}")

In [ ]:
# Condition on components from different parts of the tree simultaneously
cond3 = condition_on(nested,
    physics={"force": jnp.array(1.0)},
    observation=jnp.array(0.5),
)
print("=== Condition on force + observation ===")
print(f"Remaining components: {cond3.component_names}")
print(f"Event size: {cond3.event_size}")
print(f"Sample: {sample(cond3)}")

In [ ]:
# Equivalent positional-dict syntax
cond4 = condition_on(nested, {"physics": {"force": jnp.array(1.0)}})
print("=== Positional dict syntax ===")
print(f"Remaining components: {cond4.component_names}")

**Error handling:** You must condition on individual component
distributions, not on the intermediate dict structure that
contains them. Clear error messages guide you to the correct syntax:

In [ ]:
# Trying to condition on a dict node with a scalar raises a clear error
try:
    condition_on(nested, physics=jnp.array(1.0))
except TypeError as e:
    print(f"TypeError: {e}")

# Trying to pass a dict for a component distribution also raises
try:
    condition_on(nested, observation={"sub": jnp.array(1.0)})
except TypeError as e:
    print(f"\nTypeError: {e}")

---
## Part 3: SequentialJointDistribution

`SequentialJointDistribution` models autoregressive dependence: later
components can be callables that receive earlier samples and return
distributions.

> **Note:** Sequential joints support flat dicts only (no nesting),
> because dependencies are expressed via function parameter names.

In [ ]:
seq = SequentialJointDistribution(
    z=Normal(loc=0.0, scale=1.0),
    x=lambda z: Normal(loc=z, scale=0.5),        # x | z ~ N(z, 0.5)
    y=lambda z, x: Normal(loc=z + x, scale=0.1), # y | z,x ~ N(z+x, 0.1)
)
print(f"Components: {seq.component_names}")
print(f"Event shapes: {seq.event_shapes}")

In [ ]:
ss = sample(seq, sample_shape=(1000,))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(ss['z'], ss['x'], alpha=0.2, s=5)
axes[0].set_xlabel('z'); axes[0].set_ylabel('x')
axes[0].set_title('x tracks z (mean = z)')

axes[1].scatter(ss['z'], ss['y'], alpha=0.2, s=5)
axes[1].set_xlabel('z'); axes[1].set_ylabel('y')
axes[1].set_title('y tracks z + x ≈ 2z')

residual = ss['y'] - 2 * ss['z']
axes[2].hist(np.array(residual), bins=40, density=True)
axes[2].set_title(f'y − 2z residual (std={float(jnp.std(residual)):.2f})')
plt.tight_layout(); plt.show()

### Conditioning in Sequential Models

Conditioning on a **root** component is always sampleable. Conditioning
on a non-root whose parents are unconditioned raises `NotImplementedError`
at sample time (because forward sampling would use the prior, not the
posterior for those parents). `unnormalized_log_prob()` is always available.

In [ ]:
# Case 1: Condition on root z=2.0 — always works
cond_root = condition_on(seq, z=jnp.array(2.0))
s = sample(cond_root, sample_shape=(500,))
print(f"x|z=2 mean: {float(jnp.mean(s['x'])):.2f}  (expected ≈ 2.0)")
print(f"y|z=2 mean: {float(jnp.mean(s['y'])):.2f}  (expected ≈ 4.0)")

# Case 2: Condition on non-root x — NOT sampleable
cond_nonroot = condition_on(seq, x=jnp.array(1.0))
try:
    sample(cond_nonroot, sample_shape=(5,))
except NotImplementedError as e:
    print(f"\nCondition on non-root: NotImplementedError")

# Case 3: Condition on non-root AND all its parents — sampleable
cond_both = condition_on(seq, z=jnp.array(1.0), x=jnp.array(2.0))
s = sample(cond_both, sample_shape=(100,))
print(f"\ny|z=1,x=2 mean: {float(jnp.mean(s['y'])):.2f}  (expected ≈ 3.0)")

---
## Part 4: JointEmpirical

`JointEmpirical` stores per-component sample arrays (all with the same
number of rows) and resamples rows **jointly**, preserving correlation.

> **Note:** `JointEmpirical` supports flat dicts only.

In [ ]:
key = jax.random.PRNGKey(42)
n_samples = 500
z_samples = jax.random.normal(key, (n_samples,))
x_samples = z_samples + 0.1 * jax.random.normal(jax.random.PRNGKey(43), (n_samples,))

je = JointEmpirical(z=z_samples, x=x_samples)
print(f"Components: {je.component_names}, n={je.n}")

resampled = sample(je, sample_shape=(500,))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(np.array(z_samples), np.array(x_samples), alpha=0.3, s=5)
ax1.set_title(f'Original (corr={float(jnp.corrcoef(z_samples, x_samples)[0,1]):.2f})')
ax1.set_xlabel('z'); ax1.set_ylabel('x')
ax2.scatter(np.array(resampled['z']), np.array(resampled['x']), alpha=0.3, s=5, color='orange')
ax2.set_title(f'Resampled (corr={float(jnp.corrcoef(resampled["z"], resampled["x"])[0,1]):.2f})')
ax2.set_xlabel('z'); ax2.set_ylabel('x')
plt.tight_layout(); plt.show()

### Weighted Samples

In [ ]:
je_w = JointEmpirical(
    x=jnp.array([0.0, 1.0, 2.0, 3.0]),
    y=jnp.array([10.0, 20.0, 30.0, 40.0]),
    weights=jnp.array([0.01, 0.01, 0.08, 0.90]),
)
ws = sample(je_w, sample_shape=(1000,))
print(f"Resampled x mean: {float(jnp.mean(ws['x'])):.2f}  (expected ≈ 2.9)")
print(f"Resampled y mean: {float(jnp.mean(ws['y'])):.2f}  (expected ≈ 39)")

---
## Part 5: JointGaussian

`JointGaussian` wraps a multivariate Gaussian with named components and
cross-covariance, supporting exact analytical conditioning.

> **Note:** `JointGaussian` supports flat dicts only. Component dimensions
> are specified as integer keyword arguments.

In [ ]:
cov = jnp.array([[1.0, 0.8],
                 [0.8, 1.0]])
jg = JointGaussian(mean=jnp.array([0.0, 0.0]), cov=cov, x=1, y=1)
print(f"Components: {jg.component_names}")
print(f"Event shapes: {jg.event_shapes}")
print(f"Covariance:\n{jg.covariance}")

### Exact Conditioning

`condition_on()` applies the standard Gaussian conditioning formulas.
The result is a new `JointGaussian` over the remaining components.

In [ ]:
cond = condition_on(jg, x=jnp.array([2.0]))
print(f"Remaining: {cond.component_names}")
# μ_y|x=2 = 0 + 0.8 * 1.0 * (2 - 0) = 1.6
# σ²_y|x  = 1 - 0.8² = 0.36
print(f"Conditional mean:     {float(mean(cond)['y'][0]):.4f}  (expected 1.6)")
print(f"Conditional variance: {float(variance(cond)['y'][0]):.4f}  (expected 0.36)")

In [ ]:
jg_samples = sample(jg, sample_shape=(2000,))
cond_samples = sample(cond, sample_shape=(2000,))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(np.array(jg_samples['x'].ravel()), np.array(jg_samples['y'].ravel()),
            alpha=0.2, s=3, label='joint')
ax1.axvline(2.0, color='red', linestyle='--', label='x=2')
ax1.set_xlabel('x'); ax1.set_ylabel('y')
ax1.set_title('Joint (x, y)'); ax1.legend()

ax2.hist(np.array(cond_samples['y'].ravel()), bins=40, density=True, alpha=0.7,
         label=f"y|x=2 (μ={float(mean(cond)['y'][0]):.1f}, σ²={float(variance(cond)['y'][0]):.2f})")
ax2.set_title('Conditional y | x=2'); ax2.legend()
plt.tight_layout(); plt.show()

---
## Provenance Tracking

Every `condition_on()` call records which joint distribution was conditioned,
what was observed, and the full operation chain.

In [ ]:
cond_y = condition_on(jg, x=jnp.array([2.0]))
print(f"Operation: {cond_y.source.operation}")
print(f"Conditioned: {cond_y.source.metadata['conditioned']}")
print(f"Parent type: {type(cond_y.source.parents[0]).__name__}")

# Chain of conditioning on a SequentialJointDistribution
step1 = condition_on(seq, x=jnp.array(1.5))
step2 = condition_on(step1, z=jnp.array(0.5))
ancestors = provenance_ancestors(step2)
print(f"\nAncestors: {[type(a).__name__ for a in ancestors]}")

---
## Summary

| Feature | Flat Dict | Nested Dict |
|---------|-----------|-------------|
| Construction | `ProductArrayDistribution(x=Normal(...), y=Normal(...))` | `ProductDistribution(physics={"a": Normal(...)}, b=Normal(...))` |
| Sample type | `dict[str, Array]` | Nested `dict` of arrays |
| Component access | `joint["x"]` → `DistributionView` | `joint["physics", "a"]` → `DistributionView`; `joint["physics"]` → sub-`ProductArrayDistribution` |
| `component_names` | `("x", "y")` | `(("physics", "a"), ("b",))` |
| `condition_on` | `condition_on(joint, x=val)` | `condition_on(joint, physics={"a": val})` — per-component |
| `event_shapes` | `{"x": (), "y": (2,)}` | Nested dict of shapes |
| `flatten_value` | Concatenates components | JAX traversal order |
| Broadcasting | `DistributionView` in `workflow_function` | key paths handled automatically |